In [1]:
!pip install selenium

     --------------------------------------- 10.0/10.0 MB 10.1 MB/s eta 0:00:00
     -------------------------------------- 461.6/461.6 kB 9.6 MB/s eta 0:00:00
     ---------------------------------------- 58.3/58.3 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: sniffio
    Found existing installation: sniffio 1.2.0
    Uninstalling sniffio-1.2.0:
      Successfully uninstalled sniffio-1.2.0


In [4]:
import csv
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys

# Configurações
chrome_service = Service(executable_path='C:\ProgramData\Microsoft\Windows\Start Menu\Programs')  # Substitua pelo caminho do seu chromedriver
options = webdriver.ChromeOptions()
service = Service()
service.log_path = "chromedriver.log"
options.add_argument('--headless')  # Execute sem abrir a janela do navegador (headless)
options.add_argument('--disable-gpu')  # Necessário quando em execução no ambiente headless

# Inicializar o driver
driver = webdriver.Chrome(service=service, options=options)

# URL de login
login_url = 'https://suportehr.com.br/scp/login.php'

# Credenciais
username = 'marieliton'
password = 'ordem066'

# Iniciar o navegador e fazer login
driver.get(login_url)
time.sleep(1)  # Aguardar para garantir que a página tenha carregado

# Preencher o formulário de login
driver.find_element(By.NAME, 'userid').send_keys(username)
driver.find_element(By.NAME, 'passwd').send_keys(password)
driver.find_element(By.NAME, 'passwd').send_keys(Keys.ENTER)

time.sleep(3)  # Aguardar para garantir que o login seja concluído

# URL base
base_url = "https://suportehr.com.br/scp/tickets.php?queue=8&p="

# Número total de páginas
numero_de_paginas = 5

# Abrir o arquivo CSV para escrita
with open('dados_tickets_selenium.csv', 'w', newline='', encoding='utf-8') as csvfile:
    # Criar o objeto de escrita CSV
    csvwriter = csv.writer(csvfile)

    # Escrever o cabeçalho do CSV
    csvwriter.writerow(['Número do Chamado', 'Assunto do Chamado', 'Status do Ticket', 'Criado - Data', 'Criado - Hora', 'Encerrado - Data', 'Encerrado - Hora', 'Agente', 'Solicitante', 'E-mail', 'Departamento', 'Tópico de Ajuda'])

    # Loop através das páginas
    for pagina in range(1, numero_de_paginas + 1):
        # Construir a URL da página atual
        url = f"{base_url}{pagina}"

        # Tente obter a página, e continue tentando em caso de erro
        try:
            driver.get(url)
            time.sleep(5) # Aguardar para garantir que a página tenha carregado
        except Exception as e:
            print(f"Erro ao obter a página {pagina}. Exception: {e}")
            continue

        # Localizar e extrair os dados
        for tip_content in driver.find_elements(By.CSS_SELECTOR, 'div.tip_content'):
            chamado_numero = tip_content.find_element(By.CSS_SELECTOR, 'h2').text.split('#')[1].split(':')[0].strip()
            chamado_assunto = tip_content.find_element(By.CSS_SELECTOR, 'h2').text.split(': ')[1].strip()
            status_ticket = tip_content.find_element(By.XPATH, '//th[text()="Status do Ticket:"]/following-sibling::td').text.strip()
            data_criado, hora_criado = tip_content.find_element(By.XPATH, '//th[text()="Criado:"]/following-sibling::td').text.strip().split()

            # Se existe a informação de encerrado, extrai. Caso contrário, define como None
            encerrado_element = tip_content.find_element(By.XPATH, '//th[text()="Encerrado:"]')
            if encerrado_element:
                encerrado_full = encerrado_element.find_element(By.XPATH, './following-sibling::td').text.strip()
                data_encerrado, hora_encerrado, agente = encerrado_full.split()

                # Agente será a parte após 'by' se houver, caso contrário, será None
                try:
                    by_index = encerrado_parts.index('by')
                    agente = ' '.join(encerrado_parts[by_index + 1:])
                except ValueError:
                    agente = None
            else:
                data_encerrado, hora_encerrado, agente = None, None, None

            solicitante = tip_content.find_element(By.XPATH, '//th[text()="De:"]/following-sibling::td/a').text.strip()
            email = tip_content.find_element(By.XPATH, '//th[text()="De:"]/following-sibling::td/span[@class="faded"]').text.strip()
            departamento = tip_content.find_element(By.XPATH, '//th[text()="Departamento:"]/following-sibling::td').text.strip()
            topico_ajuda = tip_content.find_element(By.XPATH, '//th[text()="Tópico de ajuda:"]/following-sibling::td').text.strip()

            # Saída de dados para o CSV
            csvwriter.writerow([chamado_numero, chamado_assunto, status_ticket, data_criado, hora_criado,
                                data_encerrado, hora_encerrado, agente, solicitante, email, departamento, topico_ajuda])

            # Aguarde alguns segundos antes da próxima requisição
            time.sleep(1)  # Pausa por 1 segundo

# Fechar o navegador
driver.quit()

print("Execução encerrada")

Execução encerrada


In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import csv
import time

# Inicializar o driver
options = webdriver.ChromeOptions()
service = Service()
driver = webdriver.Chrome(service=service, options=options)

# URL de login
login_url = 'https://suportehr.com.br/scp/login.php'

# Credenciais
username = 'marieliton'
password = 'ordem066'

# Parâmetros de login
login_data = {
    'userid': username,
    'passwd': password
}

# Enviar requisição POST para realizar login
driver.get(login_url)
driver.find_element(By.NAME, 'userid').send_keys(username)
driver.find_element(By.NAME, 'passwd').send_keys(password)
#driver.find_element(By.NAME, 'submit').click()
driver.find_element(By.NAME, 'passwd').send_keys(Keys.ENTER)

time.sleep(3)  # Aguardar para garantir que o login seja concluído

# URL base
base_url = "https://suportehr.com.br/scp/tickets.php?queue=8&p="

# Número total de páginas
numero_de_paginas = 5

# Abrir o arquivo CSV para escrita
with open('dados_tickets.csv', 'w', newline='', encoding='utf-8') as csvfile:
    # Criar o objeto de escrita CSV
    csvwriter = csv.writer(csvfile)

    # Escrever o cabeçalho do CSV
    csvwriter.writerow(['Número do Chamado', 'Assunto do Chamado', 'Status do Ticket', 'Criado - Data', 'Criado - Hora', 'Encerrado - Data', 'Encerrado - Hora', 'Agente', 'Solicitante', 'E-mail', 'Departamento', 'Tópico de Ajuda'])

    # Loop através das páginas
    for pagina in range(1, numero_de_paginas + 1):
        # Construir a URL da página atual
        url = f"{base_url}{pagina}"

        # Acesse a URL
        driver.get(url)

        # Aguarde alguns segundos para garantir que a página seja carregada
        time.sleep(5)

        # Localize os elementos usando find_element_by_class_name ou find_element_by_tag_name
        tip_contents = driver.find_elements(By.CLASS_NAME, 'tip_content')

        # Iterar sobre as divs e extrair os dados
        for tip_content in tip_contents:
            # Extrai as informações diretamente da div
            chamado_numero = tip_content.find_element(By.TAG_NAME, 'h2').text.split('#')[1].split(':')[0].strip()
            chamado_assunto = tip_content.find_element(By.TAG_NAME, 'h2').text.split(': ')[1].strip()
            
            # Aqui estão as correções
            status_ticket = tip_content.find_element(By.TAG_NAME, 'th').text.strip()
            data_criado = tip_content.find_element(By.TAG_NAME, 'th').text.strip().split()[0]
            hora_criado = tip_content.find_element(By.TAG_NAME, 'th').text.strip().split()[1]
            
            encerrado_element = tip_content.find_element(By.TAG_NAME, 'th')
            if encerrado_element:
                encerrado_full = encerrado_element.find_element(By.TAG_NAME, 'td').text.strip()
                encerrado_parts = list(map(str.strip, encerrado_full.split(' ')))
                data_encerrado, hora_encerrado = encerrado_parts[:2]
                
                try:
                    by_index = encerrado_parts.index('by')
                    agente = ' '.join(encerrado_parts[by_index + 1:])
                except ValueError:
                    agente = None
            else:
                data_encerrado, hora_encerrado, agente = None, None, None
            
            solicitante = tip_content.find_element(By.TAG_NAME, 'th').find_element(By.TAG_NAME, 'a').text.strip()
            email = tip_content.find_element(By.TAG_NAME, 'th').find_element(By.CLASS_NAME, 'faded').text.strip()
            departamento = tip_content.find_element(By.TAG_NAME, 'th').find_element(By.TAG_NAME, 'td').text.strip()
            topico_ajuda = tip_content.find_element(By.TAG_NAME, 'th').find_element(By.TAG_NAME, 'td').text.strip()

            # Saída de dados para o CSV
            csvwriter.writerow([chamado_numero, chamado_assunto, status_ticket, data_criado, hora_criado,
                                data_encerrado, hora_encerrado, agente, solicitante, email, departamento, topico_ajuda])

            # Aguarde alguns segundos antes da próxima requisição
            time.sleep(1)

# Fechar o navegador
driver.quit()

print("Execução encerrada")


Execução encerrada
